# 08 — Deep learning training comparison

Main pipeline policy: include all age and gender groups. Age uses the ordered bands 0–6 with training-fold most-frequent filling; gender uses nominal one-hot encoding. All 148,670 rows remain. The final ensemble uses a development-selected maximum-F1 cutoff. Notebook 09 remains a historical controlled experiment; this notebook is part of the integrated 01–08 run.

## 1. Reuse the development folds

Compare three training schedules for the same MLP architecture search. The optimiser is Adam, the activation is ReLU, the initial learning rate is 0.002 and the batch size is 512.

In [1]:
from pathlib import Path
import json
import sys
import warnings
from time import perf_counter

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, ParameterGrid, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from threadpoolctl import threadpool_limits

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
cleaned = root / 'data' / 'processed' / 'loan_default_cleaned.csv'
if not cleaned.is_file():
    raise FileNotFoundError('Open Notebook 08 from the project root or its notebooks folder.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_preparation import TrainingFoldPreprocessor, split_loan_data
from src.model_tuning import SEED, BalancedTrainingPipeline
from src.model_evaluation import classification_metrics
from src.model_validation import top_fraction_predictions

tables = root / 'outputs' / 'tables' / 'model_types'
tables.mkdir(parents=True, exist_ok=True)
figures = root / 'outputs' / 'figures' / 'model_types'
figures.mkdir(parents=True, exist_ok=True)
X_dev, X_reserved, y_dev, _ = split_loan_data(pd.read_csv(cleaned))
assert X_dev.index.intersection(X_reserved.index).empty
assert len(X_dev) + len(X_reserved) == len(pd.read_csv(cleaned))
outer_folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED).split(X_dev, y_dev))
coverage = np.concatenate([validation for _, validation in outer_folds])
np.testing.assert_array_equal(np.sort(coverage), np.arange(len(X_dev)))

print(f'Development rows used: {len(X_dev):,} | Reserved rows excluded: {len(X_reserved):,}')

Development rows used: 118,936 | Reserved rows excluded: 29,734


## 2. Compare stopping rules with nested validation

With early stopping off, the MLP monitors training loss. With validation-based early stopping on, scikit-learn sets aside 10% of the fit rows and monitors validation accuracy. Grid Search still ranks configurations by Average Precision on its own validation folds. The internal stopping split is made inside the MLP after training-fold preprocessing.

[scikit-learn MLPClassifier documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

In [2]:
grid = {
    'model__hidden_layer_sizes': [(32, 16, 8), (64, 32, 16)],
    'model__alpha': [0.0001, 0.01],
    'balanced': [False, True],
}
schedules = {
    'Current': {'max_iter': 60, 'n_iter_no_change': 8, 'early_stopping': False},
    'Extended': {'max_iter': 150, 'n_iter_no_change': 30, 'early_stopping': False},
    'Extended with early stopping': {'max_iter': 150, 'n_iter_no_change': 30, 'early_stopping': True},
}
shared = {'activation': 'relu', 'solver': 'adam', 'learning_rate_init': .002, 'batch_size': 512,
          'tol': .0005, 'random_state': SEED, 'validation_fraction': .1}
display(pd.DataFrame(schedules).T)


def epochs_scorer(estimator, X, y):
    return estimator.named_steps['model'].n_iter_


records, grid_records = [], []
for fold, (fit, validation) in enumerate(outer_folds, start=1):
    assert not np.intersect1d(fit, validation).size
    X_fit, X_val = X_dev.iloc[fit], X_dev.iloc[validation]
    y_fit, y_val = y_dev.iloc[fit], y_dev.iloc[validation]
    inner = list(StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED + fold).split(X_fit, y_fit))
    for label, schedule in schedules.items():
        pipeline = BalancedTrainingPipeline([
            ('prepare', TrainingFoldPreprocessor(log_income=True)),
            ('model', MLPClassifier(**shared, **schedule)),
        ])
        search = GridSearchCV(pipeline, grid, scoring={'ap': 'average_precision', 'epochs': epochs_scorer},
                              cv=inner, refit='ap', error_score='raise', n_jobs=1)
        start = perf_counter()
        with threadpool_limits(limits=1, user_api='blas'), warnings.catch_warnings(record=True) as warning_log:
            warnings.simplefilter('always', ConvergenceWarning)
            search.fit(X_fit, y_fit)
            search_seconds = perf_counter() - start
            p = search.predict_proba(X_val)[:, 1]
            after = perf_counter()
            search.predict_proba(X_val)
            milliseconds = 1e6 * (perf_counter() - after) / len(y_val)
        cap_warnings = sum(issubclass(item.category, ConvergenceWarning) for item in warning_log)
        for item in warning_log:
            if not issubclass(item.category, ConvergenceWarning):
                warnings.warn(item.message, item.category)
        fitted = search.best_estimator_.named_steps['model']
        assert np.array_equal(search.classes_, [0, 1]) and np.isfinite(p).all()
        assert len(fitted.coefs_) == 4 and 0 < fitted.n_iter_ <= schedule['max_iter']
        results = search.cv_results_
        for i, params in enumerate(results['params']):
            grid_records.append({
                'configuration': label, 'fold': fold, 'parameters': json.dumps(params, sort_keys=True),
                'inner_split_1_ap': results['split0_test_ap'][i],
                'inner_split_2_ap': results['split1_test_ap'][i],
                'mean_inner_ap': results['mean_test_ap'][i],
                'split_1_epochs': results['split0_test_epochs'][i],
                'split_2_epochs': results['split1_test_epochs'][i],
                'mean_fit_seconds': results['mean_fit_time'][i],
                'rank': results['rank_test_ap'][i], 'selected': i == search.best_index_,
            })
        queue = top_fraction_predictions(p).astype(bool)
        caught = int(y_val.to_numpy()[queue].sum())
        threshold_metrics = classification_metrics(y_val, p >= .50)
        records.append({
            'fold': fold, 'model': label, 'ap': average_precision_score(y_val, p),
            'roc_auc': roc_auc_score(y_val, p), 'brier': brier_score_loss(y_val, p),
            'reviewed': int(queue.sum()), 'defaults_in_fold': int(y_val.sum()), 'defaults_caught': caught,
            'top10_precision': caught / queue.sum(), 'top10_recall': caught / y_val.sum(),
            'precision_050': threshold_metrics['precision'], 'recall_050': threshold_metrics['recall'],
            'f1_050': threshold_metrics['f1'], 'f2_050': threshold_metrics['f2'],
            'flagged_pct_050': threshold_metrics['flagged_pct'],
            'hidden_layers': str(fitted.hidden_layer_sizes), 'alpha': fitted.alpha,
            'balanced': search.best_params_['balanced'], 'epochs': fitted.n_iter_,
            'cap_warnings_in_search': cap_warnings, 'best_validation_accuracy': fitted.best_validation_score_,
            'search_seconds': search_seconds, 'refit_seconds': search.refit_time_,
            'prediction_ms_per_1000': milliseconds,
        })
        print(f'Fold {fold} | {label} | {fitted.n_iter_} epochs in refit | {search_seconds:.1f}s', flush=True)

candidates, deep = pd.DataFrame(grid_records), pd.DataFrame(records)
assert len(candidates) == 9 * len(list(ParameterGrid(grid)))
assert candidates.groupby(['configuration','fold']).selected.sum().eq(1).all()
assert len(deep) == 9 and not deep.duplicated(['model','fold']).any()
candidates.to_csv(tables / 'deep_grid_results.csv', index=False)
deep.to_csv(tables / 'deep_model_folds.csv', index=False)
display(deep[['model', 'fold', 'epochs', 'cap_warnings_in_search', 'ap', 'f2_050', 'search_seconds']].round(4))


,max_iter,n_iter_no_change,early_stopping
Current,60,8,False
Extended,150,30,False
Extended with early stopping,150,30,True


Fold 1 | Current | 33 epochs in refit | 88.3s


Fold 1 | Extended | 98 epochs in refit | 202.9s


Fold 1 | Extended with early stopping | 55 epochs in refit | 82.5s


Fold 2 | Current | 56 epochs in refit | 91.2s


Fold 2 | Extended | 90 epochs in refit | 200.9s


Fold 2 | Extended with early stopping | 43 epochs in refit | 79.0s


Fold 3 | Current | 45 epochs in refit | 89.6s


Fold 3 | Extended | 126 epochs in refit | 212.4s


Fold 3 | Extended with early stopping | 39 epochs in refit | 74.5s


,model,fold,epochs,cap_warnings_in_search,ap,f2_050,search_seconds
0,Current,1,33,15,0.6137,0.4297,88.2871
1,Extended,1,98,16,0.6117,0.3936,202.9372
2,Extended with early stopping,1,55,0,0.6127,0.4068,82.5023
3,Current,2,56,15,0.6153,0.6058,91.1890
4,Extended,2,90,14,0.6129,0.5853,200.8594
5,Extended with early stopping,2,43,0,0.6134,0.4283,79.0074
6,Current,3,45,15,0.6206,0.6157,89.5749
7,Extended,3,126,16,0.6216,0.5877,212.3854
8,Extended with early stopping,3,39,0,0.6158,0.3988,74.5229


## 3. Select the MLP schedule and compare model types

Select the MLP schedule by mean outer-fold AP. Recall, F1 and F2 below use 0.50 and depend on that cutoff. The top-10% queue gives a separate workload comparison with XGBoost and stacking.

Supplementary experiment. Rerun after Notebook 06 to refresh its ensemble comparison.

In [ ]:
summary = deep.groupby('model', sort=False).agg(
    mean_ap=('ap','mean'), fold_ap_sd=('ap','std'), mean_epochs=('epochs','mean'),
    cap_warnings=('cap_warnings_in_search','sum'), mean_precision_050=('precision_050','mean'),
    mean_recall_050=('recall_050','mean'), mean_f1_050=('f1_050','mean'), mean_f2_050=('f2_050','mean'),
    mean_flagged_pct_050=('flagged_pct_050','mean'), mean_search_seconds=('search_seconds','mean'),
    mean_refit_seconds=('refit_seconds','mean'), mean_prediction_ms_per_1000=('prediction_ms_per_1000','mean'),
).sort_values('mean_ap', ascending=False)
summary.to_csv(tables / 'mlp_schedule_comparison.csv')
display(summary.round(4))
winner = summary.index[0]
common_grid_choice = candidates[candidates.configuration.eq(winner)].groupby('parameters').mean_inner_ap.mean().idxmax()
selection = {'development_selected_schedule': winner, 'schedule_settings': schedules[winner],
             'shared_settings': shared, 'architecture_by_mean_inner_ap': json.loads(common_grid_choice),
             'grid': grid, 'outer_folds': 3, 'inner_folds': 2, 'ranking_metric': 'average_precision',
             'scope': 'Development experiment. MLP is separate from the frozen Notebook 07 candidates.'}
(tables / 'mlp_configuration.json').write_text(json.dumps(selection, indent=2) + '\n')

reference = pd.read_csv(root / 'outputs' / 'tables' / 'part_g' / 'fold_scores.csv')
reference = reference.loc[reference['model'].isin(['XGBoost', 'Stacking'])].copy()
assert len(reference) == 6 and not reference.duplicated(['model', 'fold']).any()
matched = pd.concat([reference, deep[reference.columns]], ignore_index=True)
assert matched.groupby('fold')['defaults_in_fold'].nunique().eq(1).all()
assert matched.groupby('fold')['reviewed'].nunique().eq(1).all()
matched.insert(1, 'model_type', matched.model.map({'XGBoost':'Traditional','Stacking':'Stacking'}).fillna('Deep learning'))
matched.to_csv(tables / 'matched_fold_scores.csv', index=False)
type_summary = matched.groupby(['model_type','model']).agg(
    mean_ap=('ap','mean'), fold_ap_sd=('ap','std'), mean_roc_auc=('roc_auc','mean'),
    mean_brier=('brier','mean'), mean_top10_recall=('top10_recall','mean'),
    defaults_caught=('defaults_caught','sum'), total_reviews=('reviewed','sum'),
).sort_values('mean_ap', ascending=False)
type_summary.to_csv(tables / 'model_type_comparison.csv')
display(type_summary.round(4))

fig, ax = plt.subplots(figsize=(7, 4), layout='constrained')
for label, rows in deep.groupby('model', sort=False):
    ax.scatter(rows.search_seconds, 100 * rows.ap, label=label)
ax.set(xlabel='Grid search time per fold (seconds)', ylabel='Outer validation AP (%)',
       title='MLP model performance and training time')
ax.legend(fontsize=8)
fig.savefig(figures / 'mlp_ap_and_training_time.png', dpi=160)
plt.show()
print(f'Development-selected MLP schedule: {winner}.')
